# 06 — Classificação via LLM (Zero-shot / Few-shot / Chain-of-Thought)

Implementa a Fase 18 do plano de elaboração: avalia o corpus de teste com
LLMs locais (Ollama/Hugging Face — `configs/llm.yaml`) sob as três
estratégias de prompt configuradas, e inspeciona qualitativamente algumas
justificativas geradas pelo modelo.

**Pré-requisito**: a etapa `labeling` já deve ter sido executada, e o
backend LLM habilitado em `configs/llm.yaml -> backends` precisa estar
acessível (Ollama local em execução, ou modelo Hugging Face baixado).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import polars as pl

from config.constants import CONFIG_FILE_NAMES
from config.paths import CONFIGS_DIR, load_project_paths
from data.loader import load_training_example_dataset
from io_utils.yaml import read_yaml
from pipelines.llm_evaluation import run_llm_evaluation_stage
from visualization.theme import apply_project_theme, save_figure

apply_project_theme()
paths = load_project_paths()
llm_config = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["llm"])

test_corpus = load_training_example_dataset(paths.test_corpus_file)
# Amostra menor: cada combinação backend x estratégia faz uma chamada de
# LLM por texto, custosa mesmo em modelos locais.
test_sample = test_corpus.sample(n=min(200, test_corpus.height), seed=42)

## Avaliação por combinação de backend e estratégia de prompt

In [ ]:
enabled_backends = [
    backend_name
    for backend_name, backend_config in llm_config["backends"].items()
    if backend_config["enabled"]
]
strategies = llm_config["prompting"]["strategies"]

comparison_rows = []
predictions_by_combination = {}
for backend_name in enabled_backends:
    for strategy in strategies:
        predictions, evaluation_result = run_llm_evaluation_stage(
            test_sample,
            backend_name=backend_name,
            strategy=strategy,
            prompt_version=llm_config["orchestration"]["prompt_template_version"],
        )
        combination_name = f"{backend_name}_{strategy}"
        predictions_by_combination[combination_name] = predictions
        comparison_rows.append({"combinacao": combination_name, **evaluation_result.point_metrics})
        print(f"{combination_name}: {evaluation_result.point_metrics}")

llm_comparison = pl.DataFrame(comparison_rows)
llm_comparison.write_csv(paths.reports_metrics_dir / "avaliacao_llm_prompting.csv")
llm_comparison

## Comparação visual entre estratégias

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(10, 5))
axis.bar(llm_comparison["combinacao"], llm_comparison["f1_macro"])
axis.set_title("F1-Macro por Combinação de Backend e Estratégia de Prompt")
axis.set_xlabel("Combinação")
axis.set_ylabel("F1-Macro")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
save_figure(figure, "comparacao_llm_prompting", directory=paths.reports_figures_dir)

## Inspeção qualitativa das justificativas

Usa `LangChainSentimentClassifier.predict_with_justification` para
observar o raciocínio do modelo em alguns exemplos — útil para entender
*por que* a estratégia vencedora acerta ou erra em casos específicos.

In [ ]:
from llm.backends import create_llm_backend
from llm.classifier import LangChainSentimentClassifier

best_backend_name = enabled_backends[0]
qualitative_classifier = LangChainSentimentClassifier(
    create_llm_backend(best_backend_name), strategy=llm_config["prompting"]["default_strategy"]
)
qualitative_classifier.fit(test_sample["text"].to_list(), test_sample["sentiment_label"].to_list())

for text, justified_output in zip(
    test_sample["text"].to_list()[:5],
    qualitative_classifier.predict_with_justification(test_sample["text"].to_list()[:5]),
    strict=True,
):
    print(f"texto: {text}")
    print(
        f"  -> sentimento={justified_output.sentimento} (confiança={justified_output.confianca:.2f})"
    )
    print(f"  -> justificativa: {justified_output.justificativa}\n")

## Conclusões

Registrar aqui: (1) qual estratégia de prompt (zero-shot/few-shot/CoT)
obteve o melhor F1-macro, e se o ganho de few-shot/CoT sobre zero-shot
compensa a latência adicional (`configs/evaluation.yaml ->
metrics.operational`); (2) padrões nas justificativas que expliquem os
principais erros (ex.: sarcasmo, negação de escopo longo); (3) como o
melhor resultado aqui se compara aos modelos supervisionados de
`notebooks/04_ml_classico.ipynb`/`05_deep_learning_transformers.ipynb` —
comparação formal em `notebooks/07_avaliacao_comparativa.ipynb`.